# Recurrentes
Lectura de datos ejemplo

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split


datos=pd.read_csv('Spiderman.csv', encoding='utf-8', index_col=0)
datos.head()

Procesamiento de datos.
Tokenizar y transformación de clases categóricas a enteros (0,1)

In [ ]:
textos=datos['Texto']
labels = datos['Clase']

#conversión de clases
labels_numerical = labels.map({'Negativo': 0, 'Positivo': 1})
y = np.array(labels_numerical)
print("etiquetas numéricas:",y)

# Tokenización: Generar un diccionario y frecuencias de texto
tokenizer = Tokenizer()
tokenizer.fit_on_texts(textos)

#Se genera un diccionario, donde a cada palabra se le asigna un número. A las palabras más utilizadas, se les asignas los valores más bajos
diccionario = tokenizer.word_index
total = len(diccionario) + 1
print("Diccionario:", diccionario)
print("Total de palabras:",total)


#Tranforma los textos en secuencias numéricas
secuencias = tokenizer.texts_to_sequences(textos)
print("Secuencias numéricas:",secuencias,'\nEjemplo de secuencias:')
print(textos.iloc[0],secuencias[0])

Proceso de padding
para el modelo, todas las secuencias deben de tener la misma longitud, se puede "rellenar" con ceros o "truncar" eliminando valores al inicio o al final de la secuencia


In [ ]:
#longitugMaxima recupera la secuencia más larga, y las demás se rellenan con 0 al final
maxlen = max(len(seq) for seq in secuencias)
X = pad_sequences(secuencias, maxlen=5, padding='post')
"""
- maxlen → longitud final de todas las secuencias.
- padding → 'pre' (relleno al inicio) o 'post' (relleno al final).
- truncating → 'pre' (recorta al inicio) o 'post' (recorta al final).
- value → número usado para rellenar (por defecto 0).
"""
print("Datos:\n",X)
# División de datos
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)

print("Distribución de clases\n:", y_train, y_test)

Modelo RNN

En keras, se hace uso de un modelo secuencia, agregando varias capas. Lo que hace que sea una red recurrente es la integración de capas tipo Embedding, SimpleRNN y Dense

In [ ]:
# Modelo RNN
model = Sequential()
"""
- SimpleRNN: más simple, pero sufre de vanishing gradient.
- LSTM: maneja dependencias largas gracias a celdas de memoria.
- GRU: más ligero que LSTM, pero con rendimiento similar.
"""
model.add(Embedding(input_dim=total, output_dim=16, input_length=maxlen))
""" Capa Embedding
Transforma índices enteros en vectores densos de valores reales que expresan relaciones semánticas
Mapea cada índice a un vector de dimensión fija (output_dim). Inicialmente los vectores son aleatorios, y se ajustan durante el entrenamiento
SALIDA: Matriz de tamaño longitudSecuenciaXoutput_dim
"""
model.add(SimpleRNN(32))
"""
Su objetivo es procesar secuencias de datos (texto, series temporales, audio)
manteniendo información del pasado mediante un estado oculto que se actualiza en cada paso de la secuencia.

"""
#model.add(Dense(32, activation='sigmoid'))
model.add(Dense(1, activation='sigmoid'))
"""
Capa totalmente conectada, permite aprender representaciones combinando todas las características de entrada.
- units: número de neuronas en la capa (dimensión de salida).
- activation: función de activación (relu, sigmoid, tanh, softmax, etc.).
- use_bias: si se incluye el término de sesgo (por defecto True).
- kernel_initializer: inicialización de los pesos (ej. glorot_uniform).
- bias_initializer: inicialización del sesgo.

"""
model.compile(loss='binary_crossentropy', metrics=['accuracy'])
# para clasificacion binaria

# Entrenamiento
model.fit(X_train, y_train, epochs=20, validation_data=(X_test, y_test), verbose=1)
#model.summary()

probTest=model.predict(X_test)
print(y_test,probTest)

#Predic ofrece probabilidades
probTrain=model.predict(X_train)
print(y_train,probTrain)

In [ ]:
from sklearn.metrics import classification_report

def Reporte(real,predict):
  claseReal=[]
  claseAsignada=[]
  for x in range(len(real)):
    if real[x]==1:
      r='Positivo'
    else:
      r='Negativo'
    if predict[x]<=0.5:
      h='Negativo'
    else:
      h='Positivo'
    claseReal.append(r)
    claseAsignada.append(h)
  print(classification_report(claseReal,claseAsignada))



Reporte(y_train,probTrain)

Reporte(y_test,probTest)